# Featuring Enginnering Using Scikit-Learn: Column Transformer and Pipeline

### Do You Want to Build a Snowman?

Let's prep some data for a model to predict the height of snowman.

<img src = "../assets/olaf.jpeg">

#### Load Packages

In [ ]:
# data analysis stack
import numpy as np
import pandas as pd

# data visualization stack
import matplotlib.pyplot as plt

%matplotlib inline
import seaborn as sns

sns.set_style("whitegrid")

# machine-learning stack
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    KBinsDiscretizer,
    PolynomialFeatures,
    FunctionTransformer,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# miscellaneous
import warnings

warnings.filterwarnings("ignore")

#### Load Data

In [ ]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data)
df_train

#### Exercise
Transform the data above using scikit-learn tools in combination with `ColumnTransformer()` and `Pipeline()` in a way that is suitable to be used for modeling
+ **Separate the DataFrame `df_train` into `X_train` and `y_train`** 
   +  Our target variable is `height_snowman_cm`
+ **Preprocess `X_train`**:
  + Identify which variables are **binary**, **categorical** and  **numeric**
  + Check which variables have **missing values**
    + **Impute missing value** as needed using appropriate strategy
  + Determine if categorical variables have **non-numeric values**
    + **Encode categorical variables** using techniques such as one-hot encoding
  + Determine if numeric variables are on different scales
    + **Scale numeric variables**
+ **Create `X_train_fe`**:
    + Once the preprocessing steps are completed, compile the transformed columns into a new DataFrame called `X_train_fe`. 


1. **Separate the DataFrame `df_train` into `X_train` and `y_train`**

In [ ]:
target = "height_snowman_cm"

In [ ]:
# Feature matrix
X_train = df_train.drop(target, axis=1)
y_train = df_train[target]

2. **Preprocess `X_train`**

In [ ]:
# Create a function to fill in missin value via interpolation
def interpolate(X, column):
    X_copy = X.copy()
    X_copy[f"{column}_interpolated"] = X_copy[[column]].interpolate()
    return X_copy[[f"{column}_interpolated"]]

We want to sequentially apply on the `dinner` column two transformations:
1. `Imputation` 
2. `Encoding`
   
Therefore we can employ the `Pipeline()` tool for it 

In [ ]:
# Define the steps of the pipeline
dinner_steps = [
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(sparse_output=False, drop="first")),
]

In [ ]:
# Instantiate the dinner pipeline object
dinner_pipeline = Pipeline(steps=dinner_steps)
dinner_pipeline

In [ ]:
trasformers = [
    (
        "temp_interpolation",
        FunctionTransformer(interpolate, kw_args={"column": "temp"}),
        ["temp"],
    ),
    ("dinner_transformation", dinner_pipeline, ["dinner"]),
    ("ohe", OneHotEncoder(sparse_output=False, drop="first"), ["lunch", "dinner"]),
]

In [ ]:
fe_column_transformer = ColumnTransformer(
    transformers=trasformers, remainder="drop"
).set_output(transform="pandas")
fe_column_transformer

In [ ]:
fe_column_transformer.fit(X_train)

In [ ]:
X_train_fe = fe_column_transformer.transform(X_train)
X_train_fe